In [ ]:
# K-00 Run config
# ACCV — Tensor Reloaded MedMNIST
# Kaggle Training Template
# Usage Instructions:
# 1. Attach the Tensor Reloaded / Multi-Task MedMNIST Kaggle dataset.
# 2. Run K-00 through K-09 from top to bottom.
# 3. Edit only this run config between experiments so results stay comparable.

RUN_NAME = "kaggle_template_v1"
REPO_NAME = "ACCV"
REPO_URL = "https://github.com/SamiiraEnache/ACCV.git"

DATASETS_TO_RUN = [
    "pathmnist",
    "dermamnist",
    "octmnist",
    "pneumoniamnist",
    "retinamnist",
    "breastmnist",
    "bloodmnist",
    "tissuemnist",
    "organamnist",
    "organcmnist",
    "organsmnist",
]

MODEL_NAME = "resnet18"
IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2

PHASE1_EPOCHS = 1
PHASE2_EPOCHS = 8
PATIENCE = 3

PHASE1_LR_HEAD = 1e-3
PHASE2_LR_BACKBONE = 1e-5
PHASE2_LR_HEAD = 1e-4
WEIGHT_DECAY = 1e-4
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 1

LOSS_TYPE = "weighted_ce"  # options: "ce", "weighted_ce", "focal"
FOCAL_GAMMA = 2.0
USE_TRAINVAL = False
TTA_ENABLED = True
SEED = 42

print("Configured run:", RUN_NAME)


In [ ]:
# K-01 Install dependencies
# Kaggle images usually include PyTorch, but timm/scipy/sklearn versions can vary.
# Installing explicitly makes this notebook reproducible across Kaggle sessions.

import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "timm",
    "scikit-learn",
    "scipy",
    "pandas",
    "tqdm",
])

print("Dependencies installed")


In [ ]:
# K-02 Clone repo
# The notebook always uses the ACCV repository as the source of truth for modules.

import os
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
REPO_DIR = WORKING_DIR / REPO_NAME
WORKING_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    print(f"Repository already exists at {REPO_DIR}")

sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
print("Current directory:", Path.cwd())


In [ ]:
# K-03 Reproducibility and output dirs
# Fixed seeds and deterministic CuDNN settings make validation comparisons easier.
# torch.__version__ is printed so every Kaggle run records its runtime context.

import json
import random
import shutil
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from scipy.stats import hmean
from torch import nn
from tqdm.auto import tqdm

from configs.base_config import BASE_CONFIG
from configs.dataset_configs import DATASET_CONFIGS, FOCUS_TUNING_DATASETS, NO_VFLIP_DATASETS, SMALL_DATASETS
from src.dataset import get_dataloaders
from src.models import build_model, freeze_backbone, get_optimizer_phase2, unfreeze_all
from src.submission import build_submission_csv
from src.train import compute_class_weights, train_one_epoch, validate
from src.transforms import get_train_transform, get_val_transform

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

DEVICE = torch.device(BASE_CONFIG["device"])
OUTPUT_DIR = Path(BASE_CONFIG["output_dir"])
SUBMISSION_DIR = Path(BASE_CONFIG["submission_dir"])
RESULTS_DIR = WORKING_DIR / "experiments"
EXPORT_DIR = WORKING_DIR / f"exports_{RUN_NAME}"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

RUN_CONFIG = {
    "run_name": RUN_NAME,
    "datasets_to_run": DATASETS_TO_RUN,
    "model_name": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "phase1_epochs": PHASE1_EPOCHS,
    "phase2_epochs": PHASE2_EPOCHS,
    "patience": PATIENCE,
    "phase1_lr_head": PHASE1_LR_HEAD,
    "phase2_lr_backbone": PHASE2_LR_BACKBONE,
    "phase2_lr_head": PHASE2_LR_HEAD,
    "weight_decay": WEIGHT_DECAY,
    "scheduler_factor": SCHEDULER_FACTOR,
    "scheduler_patience": SCHEDULER_PATIENCE,
    "loss_type": LOSS_TYPE,
    "focal_gamma": FOCAL_GAMMA,
    "use_trainval": USE_TRAINVAL,
    "tta_enabled": TTA_ENABLED,
    "seed": SEED,
}

print("torch.__version__:", torch.__version__)
print("device:", DEVICE)
print("data_root:", BASE_CONFIG["data_root"])
print(json.dumps(RUN_CONFIG, indent=2))


In [ ]:
# K-04 Verify data
# This cell fails fast if the Kaggle dataset is not attached or if any expected split is missing.
# chestmnist is intentionally excluded from this competition workflow.

required_keys = {"train_images", "train_labels", "val_images", "val_labels", "test_images"}
assert "chestmnist" not in DATASETS_TO_RUN
assert set(DATASETS_TO_RUN).issubset(set(DATASET_CONFIGS.keys()))

data_summary = []
for dataset_name in DATASETS_TO_RUN:
    npz_path = Path(BASE_CONFIG["data_root"]) / f"{dataset_name}.npz"
    assert npz_path.exists(), f"Missing dataset file: {npz_path}"
    with np.load(npz_path) as data:
        missing_keys = required_keys - set(data.files)
        assert not missing_keys, f"{dataset_name} missing keys: {missing_keys}"
        data_summary.append({
            "dataset": dataset_name,
            "train": len(data["train_images"]),
            "val": len(data["val_images"]),
            "test": len(data["test_images"]),
            "n_classes": DATASET_CONFIGS[dataset_name]["n_classes"],
            "is_rgb": DATASET_CONFIGS[dataset_name]["is_rgb"],
        })

data_summary_df = pd.DataFrame(data_summary)
display(data_summary_df)


In [ ]:
# K-05 Training loop
# Scientific methodology:
# - Phase 1 trains only the classification head so pretrained visual features are not destroyed early.
# - Phase 2 unfreezes all parameters and uses lower LR for the backbone than the head.
# - Validation macro-F1 drives checkpointing, scheduler updates, and early stopping.
# - Weighted CE and focal loss address class imbalance, which is critical for harmonic mean scoring.

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = nn.functional.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce_loss)
        return (((1.0 - pt) ** self.gamma) * ce_loss).mean()


def choose_aug_level(dataset_name):
    if dataset_name in FOCUS_TUNING_DATASETS:
        return "heavy"
    if dataset_name in SMALL_DATASETS:
        return "medium"
    return "light"


def load_train_labels(dataset_name):
    npz_path = Path(BASE_CONFIG["data_root"]) / f"{dataset_name}.npz"
    with np.load(npz_path) as data:
        return data["train_labels"].squeeze().astype(np.int64)


def make_criterion(dataset_name, n_classes):
    class_weights = compute_class_weights(load_train_labels(dataset_name), n_classes).to(DEVICE)
    if LOSS_TYPE == "weighted_ce":
        return nn.CrossEntropyLoss(weight=class_weights)
    if LOSS_TYPE == "focal":
        return FocalLoss(alpha=class_weights, gamma=FOCAL_GAMMA)
    return nn.CrossEntropyLoss()


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_f1, metadata):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "epoch": epoch,
            "best_f1": best_f1,
            "metadata": metadata,
        },
        path,
    )


def run_one_dataset(dataset_name):
    cfg = DATASET_CONFIGS[dataset_name]
    n_classes = cfg["n_classes"]
    is_rgb = cfg["is_rgb"]
    aug_level = choose_aug_level(dataset_name)
    checkpoint_path = OUTPUT_DIR / f"{RUN_NAME}_{dataset_name}_{MODEL_NAME}_best.pth"

    train_transform = get_train_transform(IMAGE_SIZE, aug_level, is_rgb, dataset_name)
    val_transform = get_val_transform(IMAGE_SIZE, is_rgb)
    train_loader, val_loader, test_loader = get_dataloaders(
        dataset_name,
        IMAGE_SIZE,
        BATCH_SIZE,
        train_transform,
        val_transform,
        num_workers=NUM_WORKERS,
        use_trainval=USE_TRAINVAL,
    )

    model = build_model(MODEL_NAME, n_classes).to(DEVICE)
    criterion = make_criterion(dataset_name, n_classes)
    best_f1 = -1.0
    best_epoch = 0
    stale_epochs = 0
    history = []

    freeze_backbone(model)
    phase1_optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=PHASE1_LR_HEAD,
        weight_decay=WEIGHT_DECAY,
    )
    phase1_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        phase1_optimizer,
        mode="max",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
    )

    for epoch in range(1, PHASE1_EPOCHS + 1):
        train_loss, train_f1 = train_one_epoch(model, train_loader, phase1_optimizer, criterion, DEVICE)
        val_loss, val_f1, per_class_f1 = validate(model, val_loader, criterion, DEVICE)
        phase1_scheduler.step(val_f1)
        history.append({
            "dataset": dataset_name,
            "phase": "phase1_head",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_macro_f1": train_f1,
            "val_loss": val_loss,
            "val_macro_f1": val_f1,
            "per_class_f1": per_class_f1.tolist(),
        })
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch
            stale_epochs = 0
            save_checkpoint(checkpoint_path, model, phase1_optimizer, phase1_scheduler, epoch, best_f1, {"dataset": dataset_name, "phase": "phase1_head", "aug_level": aug_level})
        else:
            stale_epochs += 1

    unfreeze_all(model)
    phase2_optimizer = get_optimizer_phase2(model, PHASE2_LR_BACKBONE, PHASE2_LR_HEAD, WEIGHT_DECAY)
    phase2_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        phase2_optimizer,
        mode="max",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
    )

    for local_epoch in range(1, PHASE2_EPOCHS + 1):
        epoch = PHASE1_EPOCHS + local_epoch
        train_loss, train_f1 = train_one_epoch(model, train_loader, phase2_optimizer, criterion, DEVICE)
        val_loss, val_f1, per_class_f1 = validate(model, val_loader, criterion, DEVICE)
        phase2_scheduler.step(val_f1)
        history.append({
            "dataset": dataset_name,
            "phase": "phase2_finetune",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_macro_f1": train_f1,
            "val_loss": val_loss,
            "val_macro_f1": val_f1,
            "per_class_f1": per_class_f1.tolist(),
        })
        print(f"{dataset_name} epoch={epoch} train_loss={train_loss:.4f} train_f1={train_f1:.4f} val_loss={val_loss:.4f} val_f1={val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch
            stale_epochs = 0
            save_checkpoint(checkpoint_path, model, phase2_optimizer, phase2_scheduler, epoch, best_f1, {"dataset": dataset_name, "phase": "phase2_finetune", "aug_level": aug_level})
        else:
            stale_epochs += 1

        if stale_epochs >= PATIENCE:
            print(f"Early stopping {dataset_name}: best_epoch={best_epoch}, best_f1={best_f1:.4f}")
            break

    return {
        "dataset": dataset_name,
        "checkpoint_path": str(checkpoint_path),
        "best_f1": best_f1,
        "best_epoch": best_epoch,
        "history": history,
        "test_loader": test_loader,
    }


training_outputs = {}
all_history = []
for dataset_name in DATASETS_TO_RUN:
    print("=" * 80)
    print("Training", dataset_name)
    output = run_one_dataset(dataset_name)
    training_outputs[dataset_name] = output
    all_history.extend(output["history"])

history_df = pd.DataFrame(all_history)
history_path = RESULTS_DIR / f"{RUN_NAME}_training_history.csv"
history_df.to_csv(history_path, index=False)
display(history_df.tail())


In [ ]:
# K-06 Harmonic mean
# Harmonic mean is stricter than arithmetic mean: one weak dataset lowers the final score.
# This encourages balanced performance across imaging modalities and class imbalance regimes.

val_f1_by_dataset = {name: training_outputs[name]["best_f1"] for name in DATASETS_TO_RUN}
scores = np.asarray(list(val_f1_by_dataset.values()), dtype=np.float64)
harmonic_mean_f1 = float(hmean(scores)) if np.all(scores > 0) else 0.0

results_rows = []
for dataset_name in DATASETS_TO_RUN:
    results_rows.append({
        "dataset": dataset_name,
        "model_name": MODEL_NAME,
        "image_size": IMAGE_SIZE,
        "lr_backbone": PHASE2_LR_BACKBONE,
        "lr_head": PHASE2_LR_HEAD,
        "weight_decay": WEIGHT_DECAY,
        "scheduler": "ReduceLROnPlateau",
        "aug_level": choose_aug_level(dataset_name),
        "loss_type": LOSS_TYPE,
        "epochs": training_outputs[dataset_name]["best_epoch"],
        "val_macro_f1": val_f1_by_dataset[dataset_name],
        "notes": f"checkpoint={training_outputs[dataset_name]['checkpoint_path']}",
    })

results_rows.append({
    "dataset": "harmonic_mean",
    "model_name": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "lr_backbone": PHASE2_LR_BACKBONE,
    "lr_head": PHASE2_LR_HEAD,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "aug_level": "mixed",
    "loss_type": LOSS_TYPE,
    "epochs": "",
    "val_macro_f1": harmonic_mean_f1,
    "notes": "harmonic mean across best validation macro-F1 values",
})

results_df = pd.DataFrame(results_rows)
results_path = RESULTS_DIR / f"{RUN_NAME}_results_tracker.csv"
results_df.to_csv(results_path, index=False)
print("Validation harmonic mean:", harmonic_mean_f1)
display(results_df)


In [ ]:
# K-07 Submission CSV
# TTA averages softmax probabilities across deterministic image variants.
# Horizontal flips are generally safe; vertical flips are disabled for organ/OCT datasets.
# build_submission_csv verifies task coverage and row counts against Kaggle .npz files.

def load_best_model(dataset_name):
    model = build_model(MODEL_NAME, DATASET_CONFIGS[dataset_name]["n_classes"])
    checkpoint = torch.load(training_outputs[dataset_name]["checkpoint_path"], map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(DEVICE)
    model.eval()
    return model


def predict_with_tta(model, loader, dataset_name):
    all_probabilities = []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Predict {dataset_name}"):
            images = batch[0] if isinstance(batch, (list, tuple)) else batch
            images = images.to(DEVICE)
            logits_list = [model(images)]
            if TTA_ENABLED:
                logits_list.append(model(torch.flip(images, dims=[3])))
                if dataset_name not in NO_VFLIP_DATASETS:
                    logits_list.append(model(torch.flip(images, dims=[2])))
            probabilities = torch.stack(
                [torch.softmax(logits, dim=1) for logits in logits_list],
                dim=0,
            ).mean(dim=0)
            all_probabilities.append(probabilities.cpu().numpy())
    probabilities = np.concatenate(all_probabilities, axis=0)
    return probabilities.argmax(axis=1), probabilities


test_predictions = {}
test_probabilities = {}
for dataset_name in DATASETS_TO_RUN:
    model = load_best_model(dataset_name)
    predictions, probabilities = predict_with_tta(model, training_outputs[dataset_name]["test_loader"], dataset_name)
    test_predictions[dataset_name] = predictions
    test_probabilities[dataset_name] = probabilities
    print(dataset_name, predictions.shape, probabilities.shape)

timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
submission_path = SUBMISSION_DIR / f"submission_{RUN_NAME}_{timestamp}.csv"
submission_df = build_submission_csv(test_predictions, submission_path)
assert list(submission_df.columns) == ["id", "id_image_in_task", "task_name", "label"]
assert submission_df["id"].is_monotonic_increasing
assert set(submission_df["task_name"].unique()) == set(DATASETS_TO_RUN)
print("Saved submission:", submission_path)
display(submission_df.head())


In [ ]:
# K-08 Save results/config
# Exporting config and logs next to the submission makes the Kaggle output auditable.
# Do not rely on memory or notebook state when comparing public leaderboard runs.

config_path = EXPORT_DIR / f"{RUN_NAME}_run_config.json"
dataset_config_path = EXPORT_DIR / f"{RUN_NAME}_dataset_configs.json"
probability_path = EXPORT_DIR / f"{RUN_NAME}_test_probabilities.npz"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(RUN_CONFIG, f, indent=2)

with open(dataset_config_path, "w", encoding="utf-8") as f:
    json.dump(DATASET_CONFIGS, f, indent=2)

np.savez_compressed(probability_path, **test_probabilities)

submission_log = pd.DataFrame([
    {
        "version": RUN_NAME,
        "timestamp": timestamp,
        "public_score": "",
        "key_changes": f"{MODEL_NAME}, {LOSS_TYPE}, TTA={TTA_ENABLED}",
        "notes": f"validation_harmonic_mean={harmonic_mean_f1:.6f}; submission={submission_path.name}",
    }
])
submission_log_path = RESULTS_DIR / f"{RUN_NAME}_submission_log.csv"
submission_log.to_csv(submission_log_path, index=False)

for path in [history_path, results_path, submission_log_path, submission_path]:
    shutil.copy2(path, EXPORT_DIR / Path(path).name)

print("Export directory:", EXPORT_DIR)
for path in sorted(EXPORT_DIR.iterdir()):
    print(" -", path.name)


In [ ]:
# K-09 Manual Kaggle submit
# Manual submission is intentional: inspect the CSV and exported logs before uploading.
# In Kaggle, download or submit the CSV shown below from the notebook output panel.

print("Kaggle submission file:", submission_path)
print("Results tracker:", results_path)
print("Submission log:", submission_log_path)
print("Checkpoint directory:", OUTPUT_DIR)
print("Export directory:", EXPORT_DIR)
print("Manual step: upload the submission CSV to the Kaggle competition page.")
